# Fraud Analyst: EDA dan Modeling
Notebook ini digunakan untuk Eksplorasi Data (EDA) dan Pemodelan awal untuk mendeteksi transaksi fraud.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import sys

# Log Library Versions
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Scikit-Learn version: {sklearn.__version__}")

In [ ]:
# Memuat dataset
df = pd.read_csv('../data/raw/creditcard.csv')

# Menampilkan 5 baris pertama dataset
display(df.head())

print(f"\nUkuran dataset (Baris, Kolom): {df.shape}")

## Step 1: Analisis Pola Transaksi Waktu (Time Analysis)

Kolom Time pada dataset ini mencatat detik yang berlalu sejak transaksi pertama di dalam dataset (mencakup 48 jam / 2 hari). Untuk mendapatkan *insight* yang lebih relevan dengan bisnis, kita akan mengonversi detik ini ke dalam format "Jam" (0-23) untuk melihat apakah ada perbedaan mencolok antara waktu terjadinya transaksi normal dan penipuan.

In [ ]:
# Membuat kolom baru 'Hour' dari kolom 'Time'
df['Hour'] = (df['Time'] / 3600) % 24

# Setel gaya visual seaborn
sns.set_style('whitegrid')

# Plotting distribusi
plt.figure(figsize=(12, 6))

# Histogram untuk Normal (Class = 0)
sns.histplot(data=df[df['Class'] == 0], x='Hour', stat='density', bins=24, color='blue', alpha=0.5, label='Normal', kde=True)

# Histogram untuk Fraud (Class = 1)
sns.histplot(data=df[df['Class'] == 1], x='Hour', stat='density', bins=24, color='red', alpha=0.5, label='Fraud', kde=True)

plt.title('Distribusi Kepadatan Transaksi per Jam (Normal vs Fraud)', fontsize=14, pad=15)
plt.xlabel('Jam Transaksi (0-23)', fontsize=12)
plt.ylabel('Kepadatan (Density)', fontsize=12)
plt.legend(title='Jenis Transaksi')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### 💡 Insight (Temuan Step 1):
Dari grafik distribusi di atas, kita bisa melihat bahwa transaksi **normal** (biru) memiliki ritme yang wajar, yaitu menurun tajam pada waktu malam/dini hari (terutama jam 0 hingga jam 6) ketika sebagian besar orang sedang tidur.

Namun, hal yang menarik adalah pada transaksi **Fraud** (merah), aktivitas penipuan tidak mengikuti jam tidur manusia pada umumnya. Kepadatan transaksi fraud di dini hari (misal jam 2 - 4) tidak turun serendah transaksi normal. Artinya, penipu tetap melancarkan aksinya tanpa kenal waktu.

**Rekomendasi Bisnis (Actionable Solution):**
Sistem *Anti-Fraud* harus dirancang untuk **lebih sensitif** pada transaksi yang terjadi di rentang **jam 02:00 pagi hingga 06:00 pagi**. Karena transaksi normal sangat jarang terjadi pada jam-jam tersebut, setiap lonjakan transaksi secara tiba-tiba dapat menjadi indikasi kuat adanya *Account Takeover* (pencurian kredensial akun). Bank dapat mengimplementasikan sistem yang secara otomatis memicu verifikasi lapis kedua (seperti mengirimkan kode OTP SMS atau *Push Notification* ke HP nasabah) jika ada transaksi tidak biasa di dini hari.